# 03 -- Generate DPO preference pairs

The slow one. Samples a few candidate summaries per dialogue, scores each with the composite reward, keeps the best/worst as a (chosen, rejected) pair. This is what stands in for a live PPO reward loop -- one-shot, offline, much cheaper.

Needs `01_setup_data_and_rag.ipynb` and `02_train_sft_baseline.ipynb` to have run at least once -- candidates are now sampled from the SFT checkpoint (falls back to the base model with a warning if `outputs/sft_model` isn't there yet).

*(Uses the same `GITHUB_TOKEN` Colab secret set up in notebook 01 -- see that notebook if you haven't set it up yet.)*

In [ ]:
from google.colab import drive, userdata
drive.mount('/content/drive')

import os

PROJECT_DIR = '/content/drive/MyDrive/MedAlignRL'
GITHUB_USERNAME = 'YOUR_USERNAME'   # <-- change this
GITHUB_REPO = 'MedAlignRL'         # <-- change if you named it differently

try:
    GITHUB_TOKEN = userdata.get('GITHUB_TOKEN')
except Exception:
    GITHUB_TOKEN = None
    print("No GITHUB_TOKEN secret found. Fine if your repo is public -- if it's "
          "private this clone will fail. See the setup note above.")

if GITHUB_TOKEN:
    REPO_URL = f'https://{GITHUB_TOKEN}@github.com/{GITHUB_USERNAME}/{GITHUB_REPO}.git'
else:
    REPO_URL = f'https://github.com/{GITHUB_USERNAME}/{GITHUB_REPO}.git'

if not os.path.exists(PROJECT_DIR):
    print("Cloning into Drive (first time)...")
    !git clone {REPO_URL} {PROJECT_DIR}
else:
    print("Repo already in Drive, pulling latest...")
    !cd {PROJECT_DIR} && git remote set-url origin {REPO_URL} && git fetch origin && git reset --hard origin/main

%cd {PROJECT_DIR}
!pip install -q -U -r requirements-colab.txt
!pip uninstall -y -q torchao  # Colab preinstalls an old torchao; peft raises ImportError on it during LoRA dispatch, and this repo never uses it
!pip install -q --no-deps https://s3-us-west-2.amazonaws.com/ai2-s2-scispacy/releases/v0.5.4/en_core_sci_sm-0.5.4.tar.gz  # reward.py's clinical NER needs this in every notebook that scores rewards, not just notebook 01

# en_core_sci_sm 0.5.4's config.cfg stores include_static_vectors as the
# string "False" (an old spaCy 3.7 serialization quirk); spaCy 3.8's
# stricter config validator requires an actual bool, so loading fails with
# a Config error otherwise. Patch it in place.
import importlib.util, pathlib
spec = importlib.util.find_spec("en_core_sci_sm")
for cfg in pathlib.Path(spec.origin).parent.rglob("config.cfg"):
    text = cfg.read_text()
    fixed = text.replace('include_static_vectors = "False"', 'include_static_vectors = false')
    if fixed != text:
        cfg.write_text(fixed)
        print(f"Patched {cfg}")


In [ ]:
!nvidia-smi

Start small if this is your first run -- `--n-examples 20` finishes in a few minutes and tells you the whole chain works before you commit to the full pass.

In [ ]:
%cd {PROJECT_DIR}/src
!python preference_pairs.py --n-candidates 4 --n-examples 20

### Once that looks right, do the real run

In [ ]:
!python preference_pairs.py --n-candidates 4 --n-examples 300

`../data/preference_pairs.jsonl` is now on Drive. Next: `04_train_dpo.ipynb`.